# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This notebook audits the Week-5 model the way we asked you to audit the paper: read the claim, ask where the label came from, check whether the split supports the claim, then pressure-test your own features and language.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Two constructive questions I would ask about a FlyRank paper claim:

- If the paper says pages with weaker performance signals should be prioritized for refresh, I would ask where the label is coming from and whether any of the inputs overlap the same time window as the label. If the window overlaps, the result may be partly circular rather than predictive.
- If the paper says its result generalizes, I would ask whether the validation split keeps the same client or site out of both train and test. A random row split can look stronger than it really is when repeated entities share hidden context.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 50)

RANDOM_STATE = 42

NUMERIC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

CATEGORICAL_FEATURES = [
    "competition_level",
    "content_type",
    "main_intent",
    "provider_used",
    "model_used",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
    "trend_direction",
]

MODEL_NUMERIC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

MODEL_CATEGORICAL_FEATURES = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

FEATURE_COLUMNS = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
LEAKAGE_COLUMNS = {"content_id", "client_id", "trend_direction", "trend_pct"}


def find_raw_path() -> Path:
    candidates = [
        Path("data/raw/content_refresh_anonymized.csv"),
        Path("../../data/raw/content_refresh_anonymized.csv"),
        Path("../../../data/raw/content_refresh_anonymized.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")


def precision_at_k(y_true: pd.Series, scores: np.ndarray, k: int) -> float:
    frame = pd.DataFrame({"y": y_true.to_numpy(), "score": np.asarray(scores, dtype=float)})
    if frame.empty:
        return 0.0
    top = frame.sort_values("score", ascending=False).head(min(k, len(frame)))
    return float(top["y"].mean()) if len(top) else 0.0


def metric_payload(y_true: pd.Series, scores: np.ndarray, *, prefix: str = "") -> dict[str, float]:
    probabilities = np.asarray(scores, dtype=float)
    predictions = (probabilities >= 0.5).astype(int)
    payload = {
        f"{prefix}accuracy": float(accuracy_score(y_true, predictions)),
        f"{prefix}precision": float(precision_score(y_true, predictions, zero_division=0)),
        f"{prefix}recall": float(recall_score(y_true, predictions, zero_division=0)),
        f"{prefix}f1": float(f1_score(y_true, predictions, zero_division=0)),
        f"{prefix}precision_at_20": precision_at_k(y_true, probabilities, 20),
        f"{prefix}precision_at_50": precision_at_k(y_true, probabilities, 50),
        f"{prefix}precision_at_100": precision_at_k(y_true, probabilities, 100),
    }
    if y_true.nunique() == 2:
        payload[f"{prefix}roc_auc"] = float(roc_auc_score(y_true, probabilities))
        payload[f"{prefix}average_precision"] = float(average_precision_score(y_true, probabilities))
    else:
        payload[f"{prefix}roc_auc"] = 0.0
        payload[f"{prefix}average_precision"] = 0.0
    return payload


def build_feature_matrix(frame: pd.DataFrame, *, extra_numeric: list[str] | None = None, extra_categorical: list[str] | None = None) -> tuple[pd.DataFrame, list[str]]:
    numeric_features = [column for column in MODEL_NUMERIC_FEATURES if column in frame.columns]
    if extra_numeric:
        numeric_features.extend(column for column in extra_numeric if column in frame.columns)

    categorical_features = [column for column in MODEL_CATEGORICAL_FEATURES if column in frame.columns]
    if extra_categorical:
        categorical_features.extend(column for column in extra_categorical if column in frame.columns)

    numeric_frame = frame[numeric_features].apply(pd.to_numeric, errors="coerce")
    numeric_frame = numeric_frame.replace([np.inf, -np.inf], np.nan).fillna(0)

    categorical_frame = frame[categorical_features].fillna("unknown").astype(str)
    encoded_frame = pd.get_dummies(categorical_frame, prefix=categorical_features, dummy_na=False, dtype=float)

    feature_frame = pd.concat(
        [numeric_frame.reset_index(drop=True), encoded_frame.reset_index(drop=True)],
        axis=1,
    )
    return feature_frame, list(feature_frame.columns)


def make_client_holdout_split(frame: pd.DataFrame, target: pd.Series) -> tuple[np.ndarray, np.ndarray, str]:
    all_indices = np.arange(len(frame))
    client_series = frame["client_id"].fillna("unknown").astype(str)
    unique_clients = client_series.drop_duplicates().to_numpy()

    if len(unique_clients) >= 5:
        rng = np.random.default_rng(RANDOM_STATE)
        shuffled_clients = rng.permutation(unique_clients)
        test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
        test_clients = set(shuffled_clients[:test_client_count])
        test_mask = client_series.isin(test_clients).to_numpy()
        train_indices = all_indices[~test_mask]
        test_indices = all_indices[test_mask]

        if (
            len(train_indices) > 0
            and len(test_indices) > 0
            and target.iloc[train_indices].nunique() == 2
            and target.iloc[test_indices].nunique() == 2
        ):
            return train_indices, test_indices, "client_holdout"

    train_indices, test_indices = train_test_split(
        all_indices,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=target,
    )
    return np.asarray(train_indices), np.asarray(test_indices), "stratified_row_holdout"


def make_random_split(frame: pd.DataFrame, target: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    all_indices = np.arange(len(frame))
    train_indices, test_indices = train_test_split(
        all_indices,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=target,
    )
    return np.asarray(train_indices), np.asarray(test_indices)


def build_models() -> dict[str, object]:
    return {
        "logistic_regression": Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        class_weight="balanced",
                        max_iter=1000,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "decision_tree": DecisionTreeClassifier(
            class_weight="balanced",
            max_depth=5,
            min_samples_leaf=50,
            random_state=RANDOM_STATE,
        ),
        "random_forest": RandomForestClassifier(
            class_weight="balanced_subsample",
            max_depth=10,
            min_samples_leaf=25,
            n_estimators=200,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
    }


def predict_probability(model: object, feature_frame: pd.DataFrame) -> np.ndarray:
    return np.asarray(model.predict_proba(feature_frame)[:, 1], dtype=float)


def evaluate_split(frame: pd.DataFrame, train_indices: np.ndarray, test_indices: np.ndarray) -> dict[str, dict[str, float | np.ndarray | object]]:
    feature_frame, feature_columns = build_feature_matrix(frame)
    target = frame["is_declining_label"].astype(int)
    results: dict[str, dict[str, float | np.ndarray | object]] = {}

    for model_name, model in build_models().items():
        model.fit(feature_frame.iloc[train_indices], target.iloc[train_indices])
        scores = predict_probability(model, feature_frame.iloc[test_indices])
        results[model_name] = {
            **metric_payload(target.iloc[test_indices], scores),
            "scores": scores,
            "model": model,
            "feature_columns": feature_columns,
        }

    return results


def top_feature_importance(model: object, feature_columns: list[str], limit: int = 12) -> pd.DataFrame:
    if isinstance(model, Pipeline):
        classifier = model.named_steps["model"]
        importances = np.abs(classifier.coef_[0])
    elif hasattr(model, "feature_importances_"):
        importances = np.asarray(model.feature_importances_, dtype=float)
    else:
        importances = np.zeros(len(feature_columns), dtype=float)

    return (
        pd.DataFrame({"feature": feature_columns, "importance": importances})
        .sort_values("importance", ascending=False)
        .head(limit)
        .reset_index(drop=True)
    )


def error_examples(frame: pd.DataFrame, test_indices: np.ndarray, scores: np.ndarray, limit: int = 5) -> tuple[pd.DataFrame, pd.DataFrame]:
    test_frame = frame.iloc[test_indices].copy()
    test_frame["score"] = scores
    test_frame["prediction"] = (test_frame["score"] >= 0.5).astype(int)

    columns = [
        "content_id",
        "client_id",
        "score",
        "prediction",
        "is_declining_label",
        "impressions_90d",
        "sessions_90d",
        "avg_position",
        "ctr",
        "content_age_days",
        "days_since_last_update",
        "trend_direction",
    ]

    false_negatives = (
        test_frame[(test_frame["is_declining_label"] == 1) & (test_frame["prediction"] == 0)]
        .sort_values("score", ascending=True)
        .loc[:, columns]
        .head(limit)
        .reset_index(drop=True)
    )
    false_positives = (
        test_frame[(test_frame["is_declining_label"] == 0) & (test_frame["prediction"] == 1)]
        .sort_values("score", ascending=False)
        .loc[:, columns]
        .head(limit)
        .reset_index(drop=True)
    )
    return false_negatives, false_positives


RAW_PATH = find_raw_path()
raw = pd.read_csv(RAW_PATH)

for column in NUMERIC_FEATURES:
    raw[column] = pd.to_numeric(raw[column], errors="coerce")

for column in CATEGORICAL_FEATURES:
    raw[column] = raw[column].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})

for column in NUMERIC_FEATURES:
    raw[column] = raw[column].replace([np.inf, -np.inf], np.nan).fillna(0)

frame = raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)].copy().reset_index(drop=True)
frame["is_declining_label"] = frame["trend_direction"].str.lower().eq("down").astype(int)
frame["log_impressions_90d"] = np.log1p(frame["impressions_90d"])
frame["log_clicks_90d"] = np.log1p(frame["clicks_90d"])
frame["log_sessions_90d"] = np.log1p(frame["sessions_90d"])
frame["log_ai_sessions_90d"] = np.log1p(frame["ai_sessions_90d"])

base_rate = float(frame["is_declining_label"].mean())
client_train_indices, client_test_indices, split_strategy = make_client_holdout_split(frame, frame["is_declining_label"])
print(f"Loaded {len(frame):,} rows from {RAW_PATH}")
print(f"Clients in audit frame: {frame['client_id'].nunique():,}")
print(f"Declining-label rate / base rate: {base_rate:.3f}")
print(f"Honest split strategy: {split_strategy}")
print(f"Feature columns in the final model: {len(FEATURE_COLUMNS)}")
print(f"Leakage columns excluded from the final model: {sorted(LEAKAGE_COLUMNS)}")

Loaded 30,000 rows from ../../data/raw/content_refresh_anonymized.csv
Clients in audit frame: 32
Declining-label rate / base rate: 0.542
Honest split strategy: client_holdout
Feature columns in the final model: 26
Leakage columns excluded from the final model: ['client_id', 'content_id', 'trend_direction', 'trend_pct']


## 2. My model under an honest split (before/after)

I compare the same model family under two splits: a random row split (the easy version) and a client-holdout split (the honest version). The gap between them tells me how much client memorization was helping.

In [3]:
random_train_indices, random_test_indices = make_random_split(frame, frame["is_declining_label"])

random_results = evaluate_split(frame, random_train_indices, random_test_indices)
client_results = evaluate_split(frame, client_train_indices, client_test_indices)

comparison_rows = []
for split_name, results in [("random row split", random_results), ("client holdout", client_results)]:
    best_model_name = sorted(
        results,
        key=lambda name: (
            results[name]["precision_at_50"],
            results[name]["average_precision"],
            results[name]["roc_auc"],
        ),
        reverse=True,
    )[0]
    metrics = results[best_model_name]
    comparison_rows.append(
        {
            "split": split_name,
            "best_model": best_model_name,
            "roc_auc": metrics["roc_auc"],
            "avg_precision": metrics["average_precision"],
            "precision_at_50": metrics["precision_at_50"],
            "accuracy": metrics["accuracy"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1": metrics["f1"],
        }
    )

comparison = pd.DataFrame(comparison_rows)
display(comparison.round(3))

print(f"Base rate in this frame: {base_rate:.3f}")
print("Random-split model ranking (by precision_at_50):")
random_summary = pd.DataFrame(
    {
        model_name: {
            "roc_auc": metrics["roc_auc"],
            "avg_precision": metrics["average_precision"],
            "precision_at_50": metrics["precision_at_50"],
            "accuracy": metrics["accuracy"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1": metrics["f1"],
        }
        for model_name, metrics in random_results.items()
    }
).T.sort_values("precision_at_50", ascending=False)
display(random_summary.round(3))

print("Client-holdout model ranking (by precision_at_50):")
client_summary = pd.DataFrame(
    {
        model_name: {
            "roc_auc": metrics["roc_auc"],
            "avg_precision": metrics["average_precision"],
            "precision_at_50": metrics["precision_at_50"],
            "accuracy": metrics["accuracy"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1": metrics["f1"],
        }
        for model_name, metrics in client_results.items()
    }
).T.sort_values("precision_at_50", ascending=False)
display(client_summary.round(3))

print(
    f"Best client-holdout model: {comparison.loc[comparison['split'] == 'client holdout', 'best_model'].iloc[0]}"
)
print(
    f"Precision@50 on random split vs client holdout: {comparison.loc[comparison['split'] == 'random row split', 'precision_at_50'].iloc[0]:.3f} -> {comparison.loc[comparison['split'] == 'client holdout', 'precision_at_50'].iloc[0]:.3f}"
)

,split,best_model,roc_auc,avg_precision,precision_at_50,accuracy,precision,recall,f1
0,random row split,decision_tree,0.717,0.701,0.92,0.668,0.691,0.703,0.697
1,client holdout,random_forest,0.747,0.610,0.68,0.671,0.560,0.741,0.638


Base rate in this frame: 0.542
Random-split model ranking (by precision_at_50):


,roc_auc,avg_precision,precision_at_50,accuracy,precision,recall,f1
decision_tree,0.717,0.701,0.92,0.668,0.691,0.703,0.697
logistic_regression,0.711,0.727,0.90,0.650,0.678,0.677,0.677
random_forest,0.758,0.769,0.90,0.692,0.709,0.732,0.720


Client-holdout model ranking (by precision_at_50):


,roc_auc,avg_precision,precision_at_50,accuracy,precision,recall,f1
random_forest,0.747,0.610,0.68,0.671,0.560,0.741,0.638
decision_tree,0.742,0.575,0.58,0.677,0.569,0.716,0.634
logistic_regression,0.700,0.522,0.40,0.661,0.566,0.567,0.566


Best client-holdout model: random_forest
Precision@50 on random split vs client holdout: 0.920 -> 0.680


## 3. Leakage audit

This is the attack-your-own-model step. I check that the final feature set excludes the label source, then I deliberately add the suspect columns back in and watch what happens.

In [4]:
honest_feature_frame, honest_feature_columns = build_feature_matrix(frame)
leaky_feature_frame, leaky_feature_columns = build_feature_matrix(
    frame,
    extra_numeric=["trend_pct"],
    extra_categorical=["trend_direction"],
)

leakage_overlap = sorted(set(FEATURE_COLUMNS).intersection(LEAKAGE_COLUMNS))
print(f"Leakage overlap in the final feature list: {leakage_overlap}")

leaky_comparison_rows = []
for label, feature_frame in [("honest features", honest_feature_frame), ("leaky features", leaky_feature_frame)]:
    model = build_models()["random_forest"]
    model.fit(feature_frame.iloc[client_train_indices], frame["is_declining_label"].iloc[client_train_indices])
    scores = predict_probability(model, feature_frame.iloc[client_test_indices])
    metrics = metric_payload(frame["is_declining_label"].iloc[client_test_indices], scores)
    leaky_comparison_rows.append(
        {
            "feature_set": label,
            "roc_auc": metrics["roc_auc"],
            "avg_precision": metrics["average_precision"],
            "precision_at_50": metrics["precision_at_50"],
            "accuracy": metrics["accuracy"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1": metrics["f1"],
        }
    )

leakage_comparison = pd.DataFrame(leaky_comparison_rows)
display(leakage_comparison.round(3))

leaky_model = build_models()["random_forest"]
leaky_model.fit(leaky_feature_frame.iloc[client_train_indices], frame["is_declining_label"].iloc[client_train_indices])
leaky_importances = top_feature_importance(leaky_model, leaky_feature_columns, limit=12)
print("Top features when the label-derived columns are included:")
display(leaky_importances)

honest_model = build_models()["random_forest"]
honest_model.fit(honest_feature_frame.iloc[client_train_indices], frame["is_declining_label"].iloc[client_train_indices])
honest_scores = predict_probability(honest_model, honest_feature_frame.iloc[client_test_indices])
false_negatives, false_positives = error_examples(frame, client_test_indices, honest_scores, limit=5)
print("False negatives from the honest client-holdout split:")
display(false_negatives)
print("False positives from the honest client-holdout split:")
display(false_positives)

print(
    f"Leakage check summary: precision@50 goes from {leakage_comparison.loc[leakage_comparison['feature_set'] == 'honest features', 'precision_at_50'].iloc[0]:.3f} to {leakage_comparison.loc[leakage_comparison['feature_set'] == 'leaky features', 'precision_at_50'].iloc[0]:.3f} when label-derived columns are added."
)

Leakage overlap in the final feature list: []


,feature_set,roc_auc,avg_precision,precision_at_50,accuracy,precision,recall,f1
0,honest features,0.747,0.61,0.68,0.671,0.56,0.741,0.638
1,leaky features,1.000,1.00,1.00,1.000,1.00,1.000,1.000


Top features when the label-derived columns are included:


,feature,importance
0,trend_direction_down,0.400315
1,trend_pct,0.362888
2,trend_direction_stable,0.100987
3,trend_direction_up,0.058008
4,trend_direction_new,0.017282
5,trend_direction_flat,0.010955
6,days_with_impressions,0.010663
7,log_impressions_90d,0.006799
8,avg_position,0.006224
9,content_age_days,0.003418


False negatives from the honest client-holdout split:


,content_id,client_id,score,prediction,is_declining_label,impressions_90d,sessions_90d,avg_position,ctr,content_age_days,days_since_last_update,trend_direction
0,content_28b4223f4e5f,client_98a3ab7c34,0.081668,0,1,1,1,0.0,0.00,91,1,down
1,content_34b14c00f80c,client_d4735e3a26,0.082714,0,1,3,1,0.0,0.00,308,20,down
2,content_79ac977c6e0b,client_f74efabef1,0.151622,0,1,3,1,0.7,0.00,104,8,down
3,content_472ce7ae14c0,client_d4735e3a26,0.154767,0,1,3,2,0.3,33.33,300,20,down
4,content_a55d958ec725,client_d4735e3a26,0.163946,0,1,3,1,2.7,0.00,290,20,down


False positives from the honest client-holdout split:


,content_id,client_id,score,prediction,is_declining_label,impressions_90d,sessions_90d,avg_position,ctr,content_age_days,days_since_last_update,trend_direction
0,content_d2dffcc697a4,client_f74efabef1,0.737431,1,0,5091,30,14.1,0.20,144,20,stable
1,content_00603b0349b4,client_f74efabef1,0.735212,1,0,1076,9,25.6,0.09,125,20,up
2,content_e55b8ab078b0,client_f74efabef1,0.733797,1,0,369,2,21.8,0.00,112,20,stable
3,content_f5013794ba57,client_f74efabef1,0.732604,1,0,881,2,15.7,0.00,175,20,new
4,content_ed3a7fd12cf8,client_f74efabef1,0.732316,1,0,411,5,35.2,0.00,175,20,new


Leakage check summary: precision@50 goes from 0.680 to 1.000 when label-derived columns are added.


## 4. Claim rewrite

I am rewriting my own strongest claim so it stays inside the evidence. The goal is not to sound weaker; it is to sound precise enough that someone else could audit the sentence.

In [5]:
claim_rewrites = pd.DataFrame(
    [
        {
            "overclaim": "The model clearly beats the hand-written rule at picking the right pages to review first.",
            "safe_rewrite": "On this starter slice and client-holdout split, the model ranked review candidates more precisely than the rule baseline, so it is a stronger decision-support queue for this snapshot.",
        },
        {
            "overclaim": "The top features explain why pages decline.",
            "safe_rewrite": "The top features are the strongest observed signals in this model, but they do not prove cause.",
        },
        {
            "overclaim": "High-confidence items should be refreshed automatically.",
            "safe_rewrite": "High-confidence items are the pages the model would send to review first; a human still needs to confirm context before acting.",
        },
    ]
)

display(claim_rewrites)
print("Safe language used here: observed, measured, directional, decision-support.")

,overclaim,safe_rewrite
0,The model clearly beats the hand-written rule ...,On this starter slice and client-holdout split...
1,The top features explain why pages decline.,The top features are the strongest observed si...
2,High-confidence items should be refreshed auto...,High-confidence items are the pages the model ...


Safe language used here: observed, measured, directional, decision-support.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.